In [ ]:
import pandas as pd

fcc_us = pd.read_csv("../data/raw/fcc/all_2023_202602.csv")
print(fcc_us.shape)
print(fcc_us.columns.tolist())
print(fcc_us.head(3))

In [ ]:
print("Challenges per state:")
print(fcc_us['h3_cell_state'].value_counts())

print(f"\nUnique hexagons: {fcc_us['h3_cell_id'].nunique()}")
print(f"\nOutcome breakdown:")
print(fcc_us['outcome'].value_counts())

print(f"\nProvider breakdown:")
print(fcc_us['provider_brand_name'].value_counts())

In [ ]:
import geopandas as gpd
import h3
from shapely.geometry import Polygon

def h3_to_polygon(hex_id):
    coords = h3.cell_to_boundary(hex_id)
    coords = [(lng, lat) for lat, lng in coords]
    return Polygon(coords)

fcc_us['geometry'] = fcc_us['h3_cell_id'].apply(h3_to_polygon)
fcc_geo = gpd.GeoDataFrame(fcc_us, geometry='geometry', crs='EPSG:4326')

# save
fcc_geo.to_file("../data/processed/fcc_us.gpkg", driver="GPKG")
print(f"Saved {len(fcc_geo)} records, {fcc_geo['h3_cell_id'].nunique()} unique hexagons")

In [ ]:
import geopandas as gpd
import pandas as pd
import h3
from shapely.geometry import Polygon

# load FCC data
fcc = gpd.read_file("../data/processed/fcc_us.gpkg")

# load USPS routes
usps = gpd.read_file("../data/processed/us_usps_routes_final.gpkg")

# reproject to meters
fcc_proj  = fcc.drop_duplicates(subset='h3_cell_id').to_crs(epsg=5070)
usps_proj = usps.to_crs(epsg=5070)

# spatial join — find covered hexagons
covered = gpd.sjoin(
    usps_proj[['ZIP_CRID', 'geometry']],
    fcc_proj[['h3_cell_id', 'geometry']],
    how='inner',
    predicate='intersects'
)['h3_cell_id'].unique()

# find NOT covered
not_covered = fcc_proj[~fcc_proj['h3_cell_id'].isin(covered)]
print(f"Not covered hexagons: {len(not_covered)}")
print(not_covered[['h3_cell_id', 'h3_cell_state', 'provider_brand_name', 'outcome']])

In [ ]:
import geopandas as gpd

# Census TIGER tribal land boundaries - free, no registration
tribal = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2023/AIANNH/tl_2023_us_aiannh.zip"
)
tribal = tribal.to_crs(epsg=5070)
print(f"Tribal areas loaded: {len(tribal)}")

# check which uncovered hexagons are on tribal land
tribal_join = gpd.sjoin(
    not_covered,
    tribal[['NAMELSAD', 'geometry']],
    how='left',
    predicate='intersects'
)

on_tribal = tribal_join[tribal_join['NAMELSAD'].notna()]
not_tribal = tribal_join[tribal_join['NAMELSAD'].isna()]

print(f"\nOn tribal land: {len(on_tribal)}")
print(f"Not on tribal land: {len(not_tribal)}")
print("\nTribal areas:")
print(on_tribal[['h3_cell_state', 'h3_cell_id', 'NAMELSAD']])

In [ ]:
import geopandas as gpd
import pandas as pd
import h3

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE = r"C:\Users\julak\Documents\broadband_project"
FCC_DIR = f"{BASE}/data/raw/fcc"
lte_path = f"{FCC_DIR}/bdc_26_4GLTE_mobile_broadband_h3_J25_29apr2026.gpkg"
nr_path  = f"{FCC_DIR}/bdc_26_5GNR_mobile_broadband_h3_J25_29apr2026.gpkg"
ookla_path = r"C:\Users\julak\Documents\broadband_project\data\processed\ookla\2025_Q4.gpkg"  # UPDATE this path if different

# ── 1. Load FCC H3 hex data ──────────────────────────────────────────────────
print("Loading FCC 4G LTE hexes...")
lte = gpd.read_file(lte_path)[['technology', 'mindown', 'environmnt', 'h3_res9_id', 'geometry']]
print(f"  → {len(lte):,} rows")

print("Loading FCC 5G-NR hexes...")
nr = gpd.read_file(nr_path)[['technology', 'mindown', 'environmnt', 'h3_res9_id', 'geometry']]
print(f"  → {len(nr):,} rows")

# Combine
fcc = pd.concat([lte, nr], ignore_index=True)
print(f"Combined: {len(fcc):,} rows")

# ── 2. Filter to in-vehicle coverage only ────────────────────────────────────
fcc_iv = fcc[fcc['environmnt'] == 1].copy()
print(f"In-vehicle (environmnt=1): {len(fcc_iv):,} rows")

# ── 3. Get parent H3 Res 7 for each Res 9 hex ───────────────────────────────
print("Converting Res 9 → Res 7...")
fcc_iv['h3_res7_id'] = fcc_iv['h3_res9_id'].apply(lambda x: h3.cell_to_parent(x, 7))

# Get unique Res 9 hexes that have in-vehicle coverage (any technology)
covered_res9 = fcc_iv['h3_res9_id'].unique()
print(f"Unique Res 9 hexes with in-vehicle coverage: {len(covered_res9):,}")

# ── 4. For each Res 7 hex, calculate coverage percentage ────────────────────
print("Calculating Res 7 coverage percentages...")

# Get all unique Res 7 parents
unique_res7 = fcc_iv['h3_res7_id'].unique()
print(f"Unique Res 7 hexes: {len(unique_res7):,}")

results = []
for r7 in unique_res7:
    # All Res 9 children of this Res 7 hex
    all_children = set(h3.cell_to_children(r7, 9))
    total_children = len(all_children)
    
    # How many are covered (in-vehicle)?
    covered_children = all_children.intersection(covered_res9)
    n_covered = len(covered_children)
    
    pct_covered = round(n_covered / total_children * 100, 1)
    
    results.append({
        'h3_res7_id': r7,
        'total_res9': total_children,
        'covered_res9': n_covered,
        'pct_covered': pct_covered
    })

res7_df = pd.DataFrame(results)

# ── 5. Filter partial coverage (between 0% and 100%) ────────────────────────
partial = res7_df[(res7_df['pct_covered'] > 0) & (res7_df['pct_covered'] < 100)].copy()
print(f"\nPartial coverage Res 7 hexes (0% < coverage < 100%): {len(partial):,}")
print(f"Full coverage (100%): {(res7_df['pct_covered'] == 100).sum():,}")
print(f"No coverage (0%): this data only contains covered hexes, so 0% aren't in the file")

print(f"\nPartial coverage stats:")
print(partial['pct_covered'].describe())

# ── 6. Create geometry for partial-coverage Res 7 hexes ──────────────────────
print("\nGenerating Res 7 hex geometries...")
from shapely.geometry import Polygon

def h3_to_polygon(h3_id):
    boundary = h3.cell_to_boundary(h3_id)
    # h3 returns (lat, lng), shapely needs (lng, lat)
    coords = [(lng, lat) for lat, lng in boundary]
    coords.append(coords[0])  # close the polygon
    return Polygon(coords)

partial['geometry'] = partial['h3_res7_id'].apply(h3_to_polygon)
partial_gdf = gpd.GeoDataFrame(partial, geometry='geometry', crs='EPSG:4326')

# ── 7. Cross-reference with Ookla (2025 Q4) ─────────────────────────────────
print("\nLoading Ookla 2025 Q4...")
ookla = gpd.read_file(ookla_path)
print(f"  → {len(ookla):,} tiles")

# Project both to EPSG:5070 for spatial join
partial_proj = partial_gdf.to_crs(epsg=5070)
ookla_proj = ookla.to_crs(epsg=5070)

print("Spatial join: Ookla tiles × partial-coverage Res 7 hexes...")
joined = gpd.sjoin(
    ookla_proj[['quadkey', 'avg_d_mbps', 'geometry']],
    partial_proj[['h3_res7_id', 'pct_covered', 'geometry']],
    how='inner',
    predicate='intersects'
).drop_duplicates(subset='quadkey')

print(f"Ookla tiles in partial-coverage hexes: {len(joined):,}")
print(f"Unique partial-coverage Res 7 hexes with Ookla data: {joined['h3_res7_id'].nunique()}")
print(f"Partial-coverage Res 7 hexes WITHOUT Ookla data: {len(partial) - joined['h3_res7_id'].nunique()}")

# ── 8. Save results ──────────────────────────────────────────────────────────
out_dir = f"{BASE}/data/results"
import os
os.makedirs(out_dir, exist_ok=True)

# Save all Res 7 coverage summary
res7_df.to_csv(f"{out_dir}/fcc_h3_res7_coverage_summary.csv", index=False)

# Save partial-coverage hexes
partial_gdf.to_file(f"{out_dir}/fcc_h3_res7_partial_coverage.gpkg", driver="GPKG")

# Save Ookla tiles in partial-coverage hexes
joined_gdf = gpd.GeoDataFrame(joined, geometry='geometry', crs='EPSG:5070')
joined_gdf.to_file(f"{out_dir}/ookla_in_partial_coverage_hexes.gpkg", driver="GPKG")

print(f"\nFiles saved to {out_dir}/")
print("  → fcc_h3_res7_coverage_summary.csv")
print("  → fcc_h3_res7_partial_coverage.gpkg")
print("  → ookla_in_partial_coverage_hexes.gpkg")

# ── 9. Summary table ─────────────────────────────────────────────────────────
print("\n── SUMMARY ──────────────────────────────────────────────")
print(f"Total Res 7 hexes with any in-vehicle coverage: {len(res7_df):,}")
print(f"Full coverage (100%): {(res7_df['pct_covered'] == 100).sum():,}")
print(f"Partial coverage (0-100%): {len(partial):,}")
print(f"Partial hexes WITH Ookla measurements: {joined['h3_res7_id'].nunique():,}")
print(f"Partial hexes WITHOUT Ookla measurements: {len(partial) - joined['h3_res7_id'].nunique():,}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.patches import Patch

step4 = pd.read_csv("C:/Users/julak/Documents/broadband_project/data/results/step4_state_usps_portions_complete.csv")

# fix year dtype
step4['year'] = step4['year'].astype(str)

# use 2023 Q1
q1 = step4[(step4['year'] == '2023') & (step4['quarter'] == 'Q1')].copy()
print(f"Rows: {len(q1)}")

legend_elements = [
    Patch(facecolor='#31a354', label='≥85%'),
    Patch(facecolor='#fdae6b', label='70-84%'),
    Patch(facecolor='#de2d26', label='<70%')
]

# ── Chart 1: % of ALL tiles near USPS per state ───────────────────────────────
q1_sorted = q1.sort_values('pct_all_tiles_near_usps', ascending=True)
colors = ['#de2d26' if p < 70 else '#fdae6b' if p < 85 else '#31a354'
          for p in q1_sorted['pct_all_tiles_near_usps']]

fig, ax = plt.subplots(figsize=(10, 14))
bars = ax.barh(q1_sorted['name'], q1_sorted['pct_all_tiles_near_usps'],
               color=colors, height=0.6)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8)
ax.set_title('% of All Ookla Tiles Intersecting USPS Routes — by State (2023 Q1)', fontsize=12)
ax.set_xlabel('% of All Tiles Near USPS')
ax.set_xlim(0, 110)
ax.grid(axis='x', alpha=0.3)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.savefig('C:/Users/julak/Documents/broadband_project/outputs/charts/step4_all_tiles_per_state.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved step4_all_tiles_per_state.png")

# ── Chart 2: % of SLOW tiles near USPS per state ─────────────────────────────
q1_sorted2 = q1.sort_values('pct_slow_tiles_near_usps', ascending=True)
colors2 = ['#de2d26' if p < 70 else '#fdae6b' if p < 85 else '#31a354'
           for p in q1_sorted2['pct_slow_tiles_near_usps']]

fig, ax = plt.subplots(figsize=(10, 14))
bars = ax.barh(q1_sorted2['name'], q1_sorted2['pct_slow_tiles_near_usps'],
               color=colors2, height=0.6)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8)
ax.set_title('% of Slow Ookla Tiles (<35 Mbps) Intersecting USPS Routes — by State (2023 Q1)', fontsize=12)
ax.set_xlabel('% of Slow Tiles Near USPS')
ax.set_xlim(0, 110)
ax.grid(axis='x', alpha=0.3)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.savefig('C:/Users/julak/Documents/broadband_project/outputs/charts/step4_slow_tiles_per_state.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved step4_slow_tiles_per_state.png")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.axis('off')

# ── Colors ───────────────────────────────────────────────────────────────────
c_root = '#2196F3'
c_ookla = '#4CAF50'
c_no_ookla = '#FF9800'
c_usps = '#66BB6A'
c_no_usps = '#EF5350'

def draw_box(ax, x, y, w, h, text, color, fontsize=11):
    box = mpatches.FancyBboxPatch((x - w/2, y - h/2), w, h,
                                   boxstyle="round,pad=0.3",
                                   facecolor=color, edgecolor='black', linewidth=1.5, alpha=0.9)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            fontweight='bold', color='white', wrap=True)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))

# ── Level 0: Root ────────────────────────────────────────────────────────────
draw_box(ax, 50, 88, 30, 10, '16,284\nPartial Coverage Hexes\nin Michigan', c_root, fontsize=13)

# ── Level 1: Ookla split ────────────────────────────────────────────────────
draw_arrow(ax, 40, 83, 28, 68)
draw_arrow(ax, 60, 83, 72, 68)

draw_box(ax, 28, 62, 24, 10, '1,898 (11.7%)\nHave Ookla Data', c_ookla, fontsize=12)
draw_box(ax, 72, 62, 24, 10, '14,386 (88.3%)\nNo Ookla Data', c_no_ookla, fontsize=12)

# ── Level 2: USPS split — Ookla side ────────────────────────────────────────
draw_arrow(ax, 20, 57, 14, 42)
draw_arrow(ax, 36, 57, 42, 42)

draw_box(ax, 14, 36, 22, 10, '1,844 (97.2%)\nUSPS Route\nPasses Through', c_usps, fontsize=11)
draw_box(ax, 42, 36, 22, 10, '54 (2.8%)\nNo USPS Route', c_no_usps, fontsize=11)

# ── Level 2: USPS split — No Ookla side ─────────────────────────────────────
draw_arrow(ax, 64, 57, 58, 42)
draw_arrow(ax, 80, 57, 86, 42)

draw_box(ax, 58, 36, 22, 10, '10,623 (73.8%)\nUSPS Route\nPasses Through', c_usps, fontsize=11)
draw_box(ax, 86, 36, 22, 10, '3,763 (26.2%)\nNo USPS Route', c_no_usps, fontsize=11)

# ── Bottom label ─────────────────────────────────────────────────────────────
ax.text(36, 18, 'USPS Can Measure: 12,467 hexes (76.6% of all partial-coverage hexes)',
        ha='center', va='center', fontsize=14, fontweight='bold',
        color='#2E7D32',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2))

# ── Title ────────────────────────────────────────────────────────────────────
ax.set_title('Michigan Partial-Coverage H3 Res 7 Hexes:\nMeasurement Status & USPS Feasibility',
             fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
out_dir = r"C:\Users\julak\Documents\broadband_project\data\results"
plt.savefig(f"{out_dir}/chart_tree_flow.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved to {out_dir}/chart_tree_flow.png")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

out_dir = r"C:\Users\julak\Documents\broadband_project\data\results"

state_counts = fcc.groupby('h3_cell_state').agg(
    total=('challenge_id', 'count'),
    upheld=('outcome', lambda x: x.str.contains('Upheld', case=False).sum()),
    overturned=('outcome', lambda x: x.str.contains('Overturn', case=False).sum())
).sort_values('total', ascending=False)

fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(state_counts))
ax.bar(x, state_counts['upheld'], width=0.6, label='Upheld', color='#4CAF50')
ax.bar(x, state_counts['overturned'], width=0.6, bottom=state_counts['upheld'], label='Overturned', color='#F44336')

ax.set_xticks(x)
ax.set_xticklabels(state_counts.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Number of Challenges', fontsize=12)
ax.set_title('FCC Mobile Availability Challenges by State\n(Sept 2023 – Nov 2025)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

for i, (idx, row) in enumerate(state_counts.iterrows()):
    ax.text(i, row['total'] + 1, str(int(row['total'])), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f"{out_dir}/rq1_by_state.png", dpi=300, bbox_inches='tight')
plt.show()
print("Saved!")

In [ ]:
import matplotlib.pyplot as plt

out_dir = r"C:\Users\julak\Documents\broadband_project\data\results"

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Urban/Rural
ur_counts = fcc.groupby('urban_rural')['challenge_id'].count()
axes[0].bar(ur_counts.index, ur_counts.values, color=['#2196F3', '#FF9800'])
for i, (idx, val) in enumerate(ur_counts.items()):
    axes[0].text(i, val + 5, f'{val}\n({val/len(fcc)*100:.1f}%)', ha='center', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Challenges', fontsize=12)
axes[0].set_title('Rural vs Urban', fontsize=14, fontweight='bold')

# Tribal/Non-tribal
tr_counts = fcc.groupby('tribal')['challenge_id'].count()
axes[1].bar(tr_counts.index, tr_counts.values, color=['#4CAF50', '#F44336'])
for i, (idx, val) in enumerate(tr_counts.items()):
    axes[1].text(i, val + 5, f'{val}\n({val/len(fcc)*100:.1f}%)', ha='center', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Challenges', fontsize=12)
axes[1].set_title('Tribal vs Non-tribal', fontsize=14, fontweight='bold')

plt.suptitle('FCC Mobile Availability Challenges by Demographic Indicators', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{out_dir}/rq1_demographics.png", dpi=300, bbox_inches='tight')
plt.show()

print("Chart saved!")
print(f"\n═══ SUMMARY ═══")
print(f"Urban: {ur_counts.get('Urban', 0)} challenges ({ur_counts.get('Urban', 0)/len(fcc)*100:.1f}%)")
print(f"Rural: {ur_counts.get('Rural', 0)} challenges ({ur_counts.get('Rural', 0)/len(fcc)*100:.1f}%)")
print(f"Tribal: {tr_counts.get('Tribal', 0)} challenges ({tr_counts.get('Tribal', 0)/len(fcc)*100:.1f}%)")
print(f"Non-tribal: {tr_counts.get('Non-tribal', 0)} challenges ({tr_counts.get('Non-tribal', 0)/len(fcc)*100:.1f}%)")

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

out_dir = r"C:\Users\julak\Documents\broadband_project\data\results"
raw_dir = r"C:\Users\julak\Documents\broadband_project\data\raw\census"

# ── Load and fix ─────────────────────────────────────────────────────────────
fcc = gpd.read_file(r"C:\Users\julak\Documents\broadband_project\data\processed\fcc_us.gpkg")
mask = fcc['technology'] == 'Challenge Upheld - Adjudicated by FCC'
fcc.loc[mask, 'outcome'] = 'Challenge Upheld - Adjudicated by FCC'
fcc.loc[mask, 'technology'] = None
fcc['challenge_date'] = pd.to_datetime(fcc['challenge_date'])
fcc['year_quarter'] = fcc['challenge_date'].dt.to_period('Q')

# ── Urban/Rural join ─────────────────────────────────────────────────────────
print("Urban/Rural join...")
hex_unique = fcc.dissolve(by='h3_cell_id', aggfunc={'challenge_id': 'count', 'h3_cell_state': 'first'}).reset_index()
hex_proj = hex_unique.to_crs(epsg=5070)

urban = gpd.read_file(f"zip://{raw_dir}/tl_2020_us_uac20.zip")
urban_proj = urban[['geometry']].to_crs(epsg=5070)
urban_join = gpd.sjoin(hex_proj[['h3_cell_id', 'geometry']], urban_proj, how='left', predicate='intersects')
urban_hexes = set(urban_join.dropna(subset=['index_right'])['h3_cell_id'].unique())
fcc['urban_rural'] = fcc['h3_cell_id'].apply(lambda x: 'Urban' if x in urban_hexes else 'Rural')

# ── Tribal join ──────────────────────────────────────────────────────────────
print("Tribal join...")
tribal = gpd.read_file(f"zip://{raw_dir}/tl_2025_us_aiannh.zip")
tribal_proj = tribal[['geometry']].to_crs(epsg=5070)
tribal_join = gpd.sjoin(hex_proj[['h3_cell_id', 'geometry']], tribal_proj, how='left', predicate='intersects')
tribal_hexes = set(tribal_join.dropna(subset=['index_right'])['h3_cell_id'].unique())
fcc['tribal'] = fcc['h3_cell_id'].apply(lambda x: 'Tribal' if x in tribal_hexes else 'Non-tribal')

# ── Income join ──────────────────────────────────────────────────────────────
print("Income join...")
income = pd.read_csv(
    r"C:\Users\julak\Documents\broadband_project\data\raw\census\ACSDT5Y2024.B19001-Data.csv",
    skiprows=[1]
)
income['FIPS'] = income['GEO_ID'].str.replace('1500000US', '')
income['TRACT_FIPS'] = income['FIPS'].str[:11]

bracket_cols = {
    'B19001_002E': 5000, 'B19001_003E': 12500, 'B19001_004E': 17500,
    'B19001_005E': 22500, 'B19001_006E': 27500, 'B19001_007E': 32500,
    'B19001_008E': 37500, 'B19001_009E': 42500, 'B19001_010E': 47500,
    'B19001_011E': 55000, 'B19001_012E': 67500, 'B19001_013E': 87500,
    'B19001_014E': 112500, 'B19001_015E': 137500, 'B19001_016E': 175000,
    'B19001_017E': 250000,
}
for col in bracket_cols.keys():
    income[col] = pd.to_numeric(income[col], errors='coerce')
income['B19001_001E'] = pd.to_numeric(income['B19001_001E'], errors='coerce')

def calc_median_income(row):
    total = row['B19001_001E']
    if pd.isna(total) or total == 0:
        return np.nan
    cumsum = 0
    for col, midpoint in bracket_cols.items():
        cumsum += row[col] if not pd.isna(row[col]) else 0
        if cumsum >= total / 2:
            return midpoint
    return np.nan

income['median_income'] = income.apply(calc_median_income, axis=1)
tract_income = income.groupby('TRACT_FIPS').agg(median_income=('median_income', 'median')).reset_index()
tract_income = tract_income[tract_income['median_income'] > 0]

challenge_states_fips = {
    'AL':'01','AK':'02','AZ':'04','CA':'06','CO':'08','DC':'11','DE':'10',
    'FL':'12','GA':'13','HI':'15','IA':'19','IN':'18','KS':'20','KY':'21',
    'MA':'25','MD':'24','MI':'26','MN':'27','MO':'29','MS':'28','NC':'37',
    'NY':'36','OH':'39','OR':'41','PA':'42','PR':'72','TN':'47','TX':'48',
    'VA':'51','VT':'50','WA':'53','WV':'54'
}

hex_centroids = hex_proj.copy()
hex_centroids['geometry'] = hex_centroids.geometry.centroid

print("Downloading tract boundaries...")
all_joins = []
for state_abbr, fips in challenge_states_fips.items():
    try:
        tracts = gpd.read_file(f"https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_{fips}_tract.zip")
        tracts_proj = tracts[['GEOID', 'geometry']].to_crs(epsg=5070)
        state_hexes = hex_centroids[hex_unique['h3_cell_state'] == state_abbr]
        if len(state_hexes) > 0:
            joined = gpd.sjoin(state_hexes[['h3_cell_id', 'geometry']],
                             tracts_proj[['GEOID', 'geometry']], how='left', predicate='within')
            all_joins.append(joined[['h3_cell_id', 'GEOID']])
        print(f"  ✓ {state_abbr}")
    except:
        print(f"  ✗ {state_abbr}")

tract_joins = pd.concat(all_joins, ignore_index=True).drop_duplicates(subset='h3_cell_id')
tract_joins = tract_joins.rename(columns={'GEOID': 'TRACT_FIPS'})
hex_income = tract_joins.merge(tract_income[['TRACT_FIPS', 'median_income']], on='TRACT_FIPS', how='left')

bins_inc = [0, 30000, 50000, 75000, 100000, float('inf')]
inc_labels = ['<$30K', '$30-50K', '$50-75K', '$75-100K', '>$100K']
hex_income['income_bracket'] = pd.cut(hex_income['median_income'], bins=bins_inc, labels=inc_labels)
fcc = fcc.merge(hex_income[['h3_cell_id', 'income_bracket']], on='h3_cell_id', how='left')
print("All joins done!")

# ── Common setup ─────────────────────────────────────────────────────────────
viridis = cm.get_cmap('viridis')
all_quarters = pd.period_range('2023Q1', '2025Q4', freq='Q')
q_labels = [str(q) for q in all_quarters]
x = np.arange(len(all_quarters))

# ══════════════════════════════════════════════════════════════════════════════
# CHART 1: Overall
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 6))
overall = fcc.groupby('year_quarter')['challenge_id'].count().reindex(all_quarters, fill_value=0)

ax.plot(x, overall.values, color=viridis(0.5), marker='o', markersize=6, linewidth=2.5)
ax.fill_between(x, overall.values, alpha=0.15, color=viridis(0.5))

for i, val in enumerate(overall.values):
    if val > 0:
        ax.text(i, val + 2, str(val), ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(q_labels, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Number of Challenges', fontsize=12)
ax.set_title('FCC Mobile Availability Challenges Over Time (2023–2025)', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{out_dir}/rq1_over_time_overall.png", dpi=300, bbox_inches='tight')
plt.show()
print("Chart 1 saved!")

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
fcc = gpd.read_file(
    r"C:\Users\julak\Documents\broadband_project\data\processed\fcc_us.gpkg"
)
out_dir = r"C:\Users\julak\Documents\broadband_project\data\results"

# ── 1. Load and parse income data ────────────────────────────────────────────
print("Loading income data...")

income = pd.read_csv(
    r"C:\Users\julak\Documents\broadband_project\data\raw\census\ACSDT5Y2024.B19001-Data.csv",
    skiprows=[1]
)

# Extract FIPS from GEO_ID: 1500000US010010201001 → 010010201001
income['FIPS'] = income['GEO_ID'].str.replace('1500000US', '', regex=False)
income['TRACT_FIPS'] = income['FIPS'].str[:11]
income['STATE_FIPS'] = income['FIPS'].str[:2]

# Convert ACS income columns to numeric
income_cols = [f'B19001_{i:03d}E' for i in range(1, 18)]

for col in income_cols:
    income[col] = pd.to_numeric(income[col], errors='coerce')


# Income bracket columns
bracket_cols = [
    ('B19001_002E', 0, 10000),
    ('B19001_003E', 10000, 15000),
    ('B19001_004E', 15000, 20000),
    ('B19001_005E', 20000, 25000),
    ('B19001_006E', 25000, 30000),
    ('B19001_007E', 30000, 35000),
    ('B19001_008E', 35000, 40000),
    ('B19001_009E', 40000, 45000),
    ('B19001_010E', 45000, 50000),
    ('B19001_011E', 50000, 60000),
    ('B19001_012E', 60000, 75000),
    ('B19001_013E', 75000, 100000),
    ('B19001_014E', 100000, 125000),
    ('B19001_015E', 125000, 150000),
    ('B19001_016E', 150000, 200000),
    ('B19001_017E', 200000, 250000),  # assumption for $200K+
]

# Convert to numeric
for col, _, _ in bracket_cols:
    income[col] = pd.to_numeric(income[col], errors='coerce')

income['B19001_001E'] = pd.to_numeric(income['B19001_001E'], errors='coerce')


def calc_median_income(row):
    """
    Estimate median household income from ACS B19001 grouped income data.

    Formula:
    median = L + interval * (N/2 - CF) / F
    """

    total = row['B19001_001E']

    if pd.isna(total) or total == 0:
        return np.nan

    N = total
    median_target = N / 2
    CF = 0

    for col, L, U in bracket_cols:
        F = row[col] if not pd.isna(row[col]) else 0

        if CF + F >= median_target:
            interval = U - L

            if F > 0:
                return L + interval * (median_target - CF) / F
            else:
                return L

        CF += F

    return np.nan

# ── 2. Calculate median income from grouped interval data ────────────────────


print("Calculating estimated median income...")
income['median_income'] = income.apply(calc_median_income, axis=1)


# ── 3. Aggregate to tract level ──────────────────────────────────────────────
tract_income = income.groupby('TRACT_FIPS').agg(
    total_hh=('B19001_001E', 'sum'),
    median_income=('median_income', 'median')
).reset_index()

tract_income = tract_income[tract_income['median_income'] > 0]

print(f"Tracts with income data: {len(tract_income)}")


# ── 4. Prepare unique H3 hexagons ────────────────────────────────────────────
print("\nPreparing hex centroids...")

hex_unique = fcc.dissolve(
    by='h3_cell_id',
    aggfunc={
        'challenge_id': 'count',
        'h3_cell_state': 'first'
    }
).reset_index().rename(columns={'challenge_id': 'n_challenges'})

# Project to EPSG:5070 for accurate centroid calculation
hex_proj = hex_unique.to_crs(epsg=5070)

hex_centroids = hex_proj.copy()
hex_centroids['geometry'] = hex_centroids.geometry.centroid


# ── 5. Download Census tract boundaries and spatially join hexes ─────────────
challenge_states_fips = {
    'AL': '01', 'AK': '02', 'AZ': '04', 'CA': '06', 'CO': '08',
    'DC': '11', 'DE': '10', 'FL': '12', 'GA': '13', 'HI': '15',
    'IA': '19', 'IN': '18', 'KS': '20', 'KY': '21', 'MA': '25',
    'MD': '24', 'MI': '26', 'MN': '27', 'MO': '29', 'MS': '28',
    'NC': '37', 'NY': '36', 'OH': '39', 'OR': '41', 'PA': '42',
    'PR': '72', 'TN': '47', 'TX': '48', 'VA': '51', 'VT': '50',
    'WA': '53', 'WV': '54'
}

print("Downloading tract boundaries and joining...")

all_joins = []

for state_abbr, fips in challenge_states_fips.items():
    try:
        tract_url = f"https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_{fips}_tract.zip"

        tracts = gpd.read_file(tract_url)
        tracts_proj = tracts[['GEOID', 'geometry']].to_crs(epsg=5070)

        state_hexes = hex_centroids[hex_unique['h3_cell_state'] == state_abbr]

        if len(state_hexes) > 0:
            joined = gpd.sjoin(
                state_hexes[['h3_cell_id', 'geometry']],
                tracts_proj[['GEOID', 'geometry']],
                how='left',
                predicate='within'
            )

            all_joins.append(joined[['h3_cell_id', 'GEOID']])

        print(f"  ✓ {state_abbr}")

    except Exception as e:
        print(f"  ✗ {state_abbr}: {e}")


tract_joins = pd.concat(all_joins, ignore_index=True)

tract_joins = tract_joins.drop_duplicates(subset='h3_cell_id')

tract_joins = tract_joins.rename(columns={'GEOID': 'TRACT_FIPS'})


# ── 6. Merge tract income onto H3 hexes ──────────────────────────────────────
hex_income = tract_joins.merge(
    tract_income[['TRACT_FIPS', 'median_income']],
    on='TRACT_FIPS',
    how='left'
)

print(f"\nHexes with income data: {hex_income['median_income'].notna().sum()} / {len(hex_income)}")


# ── 7. Create income brackets ────────────────────────────────────────────────
bins = [0, 30000, 50000, 75000, 100000, float('inf')]
labels = ['<$30K', '$30-50K', '$50-75K', '$75-100K', '>$100K']

hex_income['income_bracket'] = pd.cut(
    hex_income['median_income'],
    bins=bins,
    labels=labels,
    include_lowest=True
)


# ── 8. Merge income bracket back to FCC challenge data ───────────────────────
fcc = fcc.merge(
    hex_income[['h3_cell_id', 'median_income', 'income_bracket']],
    on='h3_cell_id',
    how='left'
)

print("\nIncome bracket by hexagon:")
print(hex_income['income_bracket'].value_counts().sort_index())

print("\nIncome bracket by challenge:")
print(fcc['income_bracket'].value_counts().sort_index())


# ── 9. Create chart: % breakdown of challenges per income bracket ────────────
fig, ax = plt.subplots(figsize=(10, 6))

inc_counts = fcc.groupby('income_bracket', observed=False)['challenge_id'].count()

colors = ['#F44336', '#FF9800', '#FFC107', '#8BC34A', '#4CAF50']

bars = ax.bar(
    inc_counts.index.astype(str),
    inc_counts.values,
    color=colors
)

total_challenges = inc_counts.sum()

for bar, val in zip(bars, inc_counts.values):
    pct = val / total_challenges * 100 if total_challenges > 0 else 0

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 3,
        f'{val}\n({pct:.1f}%)',
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold'
    )

ax.set_ylabel('Number of Challenges', fontsize=12)
ax.set_xlabel('Estimated Median Household Income of Census Tract', fontsize=12)
ax.set_title(
    'FCC Mobile Availability Challenges by Income Bracket',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()
plt.savefig(f"{out_dir}/rq1_income.png", dpi=300, bbox_inches='tight')
plt.show()

print("\nChart saved!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

v = plt.cm.viridis(np.linspace(0.1, 0.9, 5))
total_hex = len(fcc_unique)  # 174

# ── Chart 1: Urban/Rural ──────────────────────────────────────────────────────
fig1, ax = plt.subplots(figsize=(6, 5))

urban_counts = urban_join['urban_rural'].value_counts()
labels = ['Rural', 'Urban']
counts = [urban_counts.get('Rural', 0), urban_counts.get('Urban', 0)]
pcts   = [c / total_hex * 100 for c in counts]
colors = [v[0], v[4]]

bars = ax.bar(labels, pcts, color=colors, width=0.4)
for bar, count, pct in zip(bars, counts, pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{count}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Urban vs Rural Distribution\nof FCC MAC Challenge Hexagons', fontsize=12)
ax.set_ylabel('% of Hexagons')
ax.set_ylim(0, 85)
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('C:/Users/julak/Documents/broadband_project/outputs/charts/mac_urban_rural.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved mac_urban_rural.png")

# ── Chart 2: Income bracket ───────────────────────────────────────────────────
fig2, ax = plt.subplots(figsize=(7, 5))

income_counts = tract_join['income_bracket'].value_counts().sort_index()
income_labels = income_counts.index.astype(str).tolist()
counts2 = income_counts.values.tolist()
pcts2   = [c / total_hex * 100 for c in counts2]

bars = ax.bar(income_labels, pcts2, color=v, width=0.5)
for bar, count, pct in zip(bars, counts2, pcts2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{count}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Income Bracket Distribution\nof FCC MAC Challenge Hexagons', fontsize=12)
ax.set_ylabel('% of Hexagons')
ax.set_ylim(0, 52)
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('C:/Users/julak/Documents/broadband_project/outputs/charts/mac_income_bracket.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved mac_income_bracket.png")